<a href="https://colab.research.google.com/github/willianbayer22/willianbayer22/blob/main/E_I_3_Ci%C3%AAncias_de_Dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import requests
import pandas as pd

from datetime import datetime
from zoneinfo import ZoneInfo
from google.colab import files

In [19]:
# Moedas que serão consultadas.
MOEDAS = {
    "USD": "Dólar americano",
    "EUR": "Euro",
    "GBP": "Libra esterlina"
}

URL_BASE = "https://api.frankfurter.dev/v2/rate"


def consultar_cotacoes():
    """Consulta as moedas e retorna uma tabela com as cotações."""
    registros = []

    data_coleta = datetime.now(
        ZoneInfo("America/Sao_Paulo")
    ).isoformat(timespec="seconds")

    for codigo, nome in MOEDAS.items():
        url = f"{URL_BASE}/{codigo}/BRL"

        resposta = requests.get(url, timeout=30)
        resposta.raise_for_status()

        dados = resposta.json()

        registros.append({
            "moeda": codigo,
            "nome": nome,
            "moeda_destino": "BRL",
            "valor_em_reais": float(dados["rate"]),
            "data_cotacao": dados["date"],
            "data_coleta": data_coleta,
            "fonte": "Frankfurter"
        })

    return pd.DataFrame(registros)


# Evita usar uma tabela antiga caso uma nova consulta falhe.
tabela = None

try:
    tabela = consultar_cotacoes()
except (requests.RequestException, ValueError, KeyError) as erro:
    raise RuntimeError(
        f"Não foi possível obter as cotações: {erro}"
    ) from erro

print("Cotações coletadas com sucesso!")
display(tabela)

Cotações coletadas com sucesso!


,moeda,nome,moeda_destino,valor_em_reais,data_cotacao,data_coleta,fonte
0,USD,Dólar americano,BRL,5.6309,2026-11-14,2026-11-14T16:52:12-03:00,Frankfurter
1,EUR,Euro,BRL,6.2403,2026-11-14,2026-11-14T16:52:12-03:00,Frankfurter
2,GBP,Libra esterlina,BRL,7.4683,2026-11-14,2026-11-14T16:52:12-03:00,Frankfurter


In [20]:
if tabela is None or tabela.empty:
    raise RuntimeError("Execute a coleta antes de salvar os arquivos.")

# CSV: tabela que pode ser aberta no Excel.
tabela.to_csv(
    "cotacoes.csv",
    index=False,
    sep=";",
    decimal=",",
    encoding="utf-8-sig"
)

print("Arquivos cotacoes.csv criado!")

Arquivos cotacoes.csv criado!


In [21]:
files.download("cotacoes.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>